# ⚽ WC2026 Predictor — Análisis Exploratorio de Datos (EDA)

Este notebook explora el dataset histórico de partidos internacionales y los ratings Elo calculados,
documentando los hallazgos clave que guiaron el diseño del modelo.

**Hallazgos principales:**
- La Regresión Logística supera consistentemente a redes neuronales y modelos de árboles
- Las variables xG del torneo actual son ruido estadístico (muestra insuficiente: 4-8 partidos)
- El Elo y la forma histórica son las señales más robustas y persistentes

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

sns.set_theme(style='whitegrid', palette='muted')
DATA_DIR = Path('../data')
print('✅ Librerías cargadas')

## 1. Dataset histórico

In [ ]:
df = pd.read_csv(DATA_DIR / 'results.csv')
df['date'] = pd.to_datetime(df['date'])
print(f'Total partidos: {len(df):,}')
print(f'Rango: {df["date"].min().date()} → {df["date"].max().date()}')
print(f'Torneos únicos: {df["tournament"].nunique()}')
df.head()

## 2. Distribución histórica de resultados

In [ ]:
played = df.dropna(subset=['home_score', 'away_score']).copy()
played['home_score'] = played['home_score'].astype(int)
played['away_score'] = played['away_score'].astype(int)

def result(r):
    if r['home_score'] > r['away_score']: return 'Gana Local'
    if r['home_score'] == r['away_score']: return 'Empate'
    return 'Gana Visitante'

played['result'] = played.apply(result, axis=1)

# Global
dist_global = played['result'].value_counts(normalize=True) * 100

# Solo Copa del Mundo
wc = played[(played['tournament']=='FIFA World Cup')]
dist_wc = wc['result'].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
dist_global.plot(kind='bar', ax=axes[0], color=['#1B5E20','#F57F17','#C62828'], rot=0)
axes[0].set_title('Distribución global (todos los torneos)', fontweight='bold')
axes[0].set_ylabel('% de partidos')
for p in axes[0].patches:
    axes[0].annotate(f'{p.get_height():.1f}%', (p.get_x()+0.1, p.get_height()+0.3))

dist_wc.plot(kind='bar', ax=axes[1], color=['#1B5E20','#F57F17','#C62828'], rot=0)
axes[1].set_title('Distribución Copa del Mundo', fontweight='bold')
for p in axes[1].patches:
    axes[1].annotate(f'{p.get_height():.1f}%', (p.get_x()+0.1, p.get_height()+0.3))

plt.tight_layout()
plt.savefig('../reports/distribucion_resultados.png', dpi=150, bbox_inches='tight')
plt.show()
print(dist_global)

## 3. Evolución histórica de los top 10 ratings Elo

In [ ]:
from elo import compute_elo
engine, snapshots = compute_elo(DATA_DIR / 'results.csv')

# Top 10 selecciones actuales
top10 = engine.current_table().head(10)['team'].tolist()
print('Top 10 Elo actual:', top10)

hist = engine.history_df()
hist['date'] = pd.to_datetime(hist['date'])
hist_top = hist[hist['team'].isin(top10) & (hist['date'] >= '2010-01-01')]

fig, ax = plt.subplots(figsize=(14, 6))
for team in top10:
    d = hist_top[hist_top['team']==team].set_index('date')['elo']
    ax.plot(d.index, d.values, label=team, linewidth=1.5)

ax.set_title('Evolución del Rating Elo — Top 10 selecciones (2010–2026)', fontweight='bold')
ax.set_ylabel('Rating Elo'); ax.legend(loc='upper left', fontsize=8)
ax.axvline(pd.Timestamp('2026-06-11'), color='red', linestyle='--', alpha=0.5, label='WC 2026')
plt.tight_layout()
plt.savefig('../reports/elo_evolution.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlación de features con el resultado

In [ ]:
feat_path = DATA_DIR / 'train_features.csv'
if feat_path.exists():
    feat_df = pd.read_csv(feat_path)
    feat_df['result_num'] = feat_df['result'].map({'H': 1, 'D': 0, 'A': -1})

    num_cols = ['elo_diff', 'elo_ratio', 'we_h', 'form_diff', 'streak_diff',
                'gd_diff', 'draw_risk', 'h2h', 'conf_diff', 'wc_diff']
    corr = feat_df[num_cols + ['result_num']].corr()['result_num'].drop('result_num').sort_values(ascending=False)

    fig, ax = plt.subplots(figsize=(10, 5))
    colors = ['#1B5E20' if v > 0 else '#C62828' for v in corr.values]
    corr.plot(kind='barh', ax=ax, color=colors)
    ax.set_title('Correlación de features con resultado (H=+1, D=0, A=-1)', fontweight='bold')
    ax.axvline(0, color='black', linewidth=0.8)
    plt.tight_layout()
    plt.savefig('../reports/feature_correlation.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(corr)
else:
    print('⚠️  Ejecuta primero: python src/features.py')

## 5. Accuracy del modelo por ronda — evolución del proyecto

In [ ]:
rondas = {
    'J1 Grupos (v1)':  (17, 24),
    'J2 Grupos (v1)':  (17, 24),
    'J3 Grupos (v2)':  (18, 28),
    '16avos (v3)':     (12, 16),
    'Octavos (v4)':    (6,   8),
    'Cuartos (v5)':    (4,   4),
}

labels   = list(rondas.keys())
accuracy = [a/t*100 for a,t in rondas.values()]
aciertos = [a for a,_ in rondas.values()]
totales  = [t for _,t in rondas.values()]

fig, ax = plt.subplots(figsize=(11, 5))
bars = ax.bar(labels, accuracy,
              color=['#C8E6C9' if a==100 else '#1F3864' for a in accuracy],
              edgecolor='white', linewidth=0.5)

for bar, ac, tot, pct in zip(bars, aciertos, totales, accuracy):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
            f'{ac}/{tot}\n{pct:.1f}%', ha='center', va='bottom',
            fontweight='bold', fontsize=9)

ax.axhline(70, color='orange', linestyle='--', alpha=0.7, label='70% referencia')
ax.set_ylim(0, 115)
ax.set_ylabel('Accuracy (%)')
ax.set_title('Accuracy por ronda — WC2026 Predictor', fontweight='bold', fontsize=13)
ax.legend()
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig('../reports/accuracy_por_ronda.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Comparación de modelos (walk-forward)

In [ ]:
model_comp_path = DATA_DIR / 'model_comparison.csv'
if model_comp_path.exists():
    mc = pd.read_csv(model_comp_path).sort_values('LL_mean')
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    colors = ['#1B5E20' if i < 3 else '#9E9E9E' for i in range(len(mc))]

    axes[0].barh(mc['Modelo'], mc['Acc_mean'], xerr=mc['Acc_std'],
                 color=colors, capsize=4)
    axes[0].set_title('Accuracy (walk-forward 5 folds)', fontweight='bold')
    axes[0].set_xlabel('Accuracy')

    axes[1].barh(mc['Modelo'], mc['LL_mean'], xerr=mc['LL_std'],
                 color=colors, capsize=4)
    axes[1].set_title('Log-Loss (menor = mejor)', fontweight='bold')
    axes[1].set_xlabel('Log-Loss')

    plt.suptitle('Comparación de modelos — WC2026 Predictor v6', fontweight='bold')
    plt.tight_layout()
    plt.savefig('../reports/model_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print(mc.to_string(index=False))
else:
    print('⚠️  Ejecuta primero: python src/train.py')

## 7. Conclusiones

### Sobre el modelo
- **Regresión Logística calibrada** es el mejor modelo consistentemente (log-loss ~0.87)
- **Redes neuronales (MLP)** obtienen log-loss 3.3–4.1 — el peor resultado. El fútbol tiene demasiado ruido para justificar arquitecturas complejas con ~9,600 muestras
- El **ensemble de los top-3** produce probabilidades más estables y mejor calibradas

### Sobre las features
- **`we_h`** (probabilidad Elo esperada) es la variable #1 — captura toda la fuerza relativa en un número
- Las **variables xG del torneo** son sistemáticamente descartadas por Permutation Importance
- El modelo converge a **20 variables** de alta señal: Elo, forma, confederación, H2H y racha

### Sobre el fútbol
- Los **empates son el mayor reto** de predicción (~23% de partidos pero difíciles de anticipar)
- Los **penales son azar puro** (~50/50) — cualquier modelo fallará en partidos decididos así
- El **accuracy sostenido de 70-75%** en rondas KO es muy competitivo para este dominio